# 🎮 Minecraft Server 26.3 — v9 PERMANENT + DOUBLE FALLBACK ✅

**Sirf ▶️ Run all dabao — bas! Baaki SAB automatic hai.**

## 🛡️ v9 me har cheez ka backup hai:

| Cheez | Primary | Fallback (automatic) |
|---|---|---|
| **🌐 Panel link** | ngrok (hamesha wahi URL) | Cloudflare backup link (takrav-proof) |
| **☕ Java 25** | Adoptium JRE | JDK → Drive cache → Azul |
| **📦 server.jar** | Paper API (latest) | Drive cache se |
| **💾 World** | local save | Google Drive backup (har 5 min) |
| **🤖 Server/panel/tunnels** | auto-restart watchdog (30 sec) | — |

## 🔗 PANEL ke 2 links milenge (dono chalte hain):
1. **PERMANENT** — `tidal-chug-rasping.ngrok-free.dev` (hamesha yahi, bookmark karo)
   - Agar koi **purana session** isse rok raha ho to ⏳ dikhega — **kuch karna nahi**, wo khud band hote hi ye KHUD live ho jayega
2. **BACKUP** — Cloudflare wala link, **har haal me turant chalta hai** 🛟

**Kaise chalana hai:**
1. 🔑 **Ek baar** secrets add karo — left sidebar ka 🔑 icon → `NGROK_TOKEN` + `PLAYIT_SECRET` (values chat me di hain) → **Notebook access** toggle ON
2. ▶️ **Run all** dabao
3. Google Drive ka **popup aaye → ALLOW** karo (1 baar — world + cache ke liye)
4. **Last cell "KEEPALIVE" hamesha chalta rahega — ye normal hai!** ✅

> 🔑 Ye file **PUBLIC** hai aur isme **koi secret nahi** hai — secrets Colab ke 🔑 me rehte hain (1 baar daalo, hamesha yaad rahenge). **100% safe.**

**⚠️ ZAROORI:**
- **Ye tab khula rakho** (phone lock/background OK)
- Colab free limit ~12 ghante — uske baad wapas **Run all** (world Drive se wapas aa jayega)
- Is file me koi secret nahi hai — sab Colab 🔑 me safe hai ✅

In [ ]:
!sudo apt-get update -qq && sudo apt-get install -y -qq screen wget curl rsync procps psmisc
!pip install -q flask psutil
print('✅ Base deps installed!')

In [ ]:
# 🔑 SECRETS (ONE-TIME setup — Colab hamesha yaad rakhega)
# Kaise: left sidebar me 🔑 (key) icon → "Add new secret"
#   Name: NGROK_TOKEN    | Value: chat me diya hai (permanent panel link ke liye)
#   Name: PLAYIT_SECRET  | Value: chat me diya hai (Minecraft ke liye — ZAROORI)
# ⚠️ Dono ke "Notebook access" toggle ON karna!
from google.colab import userdata

NGROK_TOKEN = ''
PLAYIT_SECRET = ''
try:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN') or ''
except Exception:
    pass
try:
    PLAYIT_SECRET = userdata.get('PLAYIT_SECRET') or ''
except Exception:
    pass

if NGROK_TOKEN and PLAYIT_SECRET:
    print('✅ Secrets mil gaye — ngrok + playit dono ready!')
else:
    print('⚠️' * 18)
    if not NGROK_TOKEN:
        print('❌ NGROK_TOKEN nahi mila!')
        print('   Sidebar 🔑 → Add new secret → Name: NGROK_TOKEN')
        print('   Value chat me hai | "Notebook access" toggle ON karo')
    if not PLAYIT_SECRET:
        print('❌ PLAYIT_SECRET nahi mila!')
        print('   Sidebar 🔑 → Add new secret → Name: PLAYIT_SECRET')
        print('   Value chat me hai | "Notebook access" toggle ON karo')
    print('   ℹ️ Sirf EK BAAR karna hai — phir hamesha yaad rahega.')
    print('   ℹ️ In dono ke bina: permanent panel link + Minecraft nahi chalega.')
    print('⚠️' * 18)

In [ ]:
# 🧹 AUTO-CLEANUP — is session ke purane sab processes band (re-run safe)
import subprocess, time, os

subprocess.run("pkill -f 'ngrok' 2>/dev/null || true", shell=True, capture_output=True)
subprocess.run("pkill -f 'cloudflared' 2>/dev/null || true", shell=True, capture_output=True)
subprocess.run("pkill -f 'panel.py' 2>/dev/null || true", shell=True, capture_output=True)
subprocess.run("pkill -x playit 2>/dev/null || true", shell=True, capture_output=True)
subprocess.run("pkill -f 'server.jar' 2>/dev/null || true", shell=True, capture_output=True)
subprocess.run("fuser -k 8081/tcp >/dev/null 2>&1 || true", shell=True)
subprocess.run("fuser -k 4040/tcp >/dev/null 2>&1 || true", shell=True)
subprocess.run(['screen', '-S', 'mcserver', '-X', 'quit'], capture_output=True)
subprocess.run(['screen', '-wipe'], capture_output=True)
for f in ('STOP_FLAG', 'NGROK_URL', 'CF_URL'):
    if os.path.exists(f):
        os.remove(f)
time.sleep(2)
print('✅ Cleanup done — clean start!')

In [ ]:
# 💾 GOOGLE DRIVE — world backup + Java/jar cache
import os, subprocess

DRIVE_BASE = None
try:
    if os.path.exists('/content/drive/MyDrive'):
        print('💾 Drive pehle se connected hai')
    else:
        from google.colab import drive
        print('⏳ Google Drive connect ho raha hai...')
        print('   👉 POPUP aaye to apna account choose karke ALLOW karo (sirf 1 baar)')
        drive.mount('/content/drive')
    base = '/content/drive/MyDrive/MC_Colab'
    os.makedirs(base, exist_ok=True)
    DRIVE_BASE = base
    print('💾 Drive connected:', base, '(world + caches yahan save honge)')
except Exception as e:
    print('⚠️ Drive connect nahi hua:', e)
    print('   ℹ️ Koi baat nahi — sab chalega! Bas world sirf local rahega aur cache nahi milega.')

# Purana world Drive me hai to wapas laao (naye session me)
if DRIVE_BASE:
    for d in ('world', 'world_nether', 'world_the_end'):
        src = os.path.join(DRIVE_BASE, d)
        dst = '/content/' + d
        if os.path.isdir(src):
            if not os.path.isdir(dst):
                subprocess.run(['rsync', '-a', src + '/', dst + '/'], capture_output=True)
                print('✅ Wapas aaya (Drive se):', d)
            else:
                print('ℹ️ Local me pehle se hai:', d)
    print('💾 World backup ON — har 5 min Drive me save hoga')
else:
    print('ℹ️ World backup OFF (sirf is session ke liye chalega)')

In [ ]:
# ☕ JAVA 25 (Paper 26.3 ke liye ZAROORI) — 4-step fallback chain
import urllib.request, subprocess, os, re, shutil

UA = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
JAVA = '/opt/java25/bin/java'

def java_ok():
    if not os.path.exists(JAVA):
        return False
    try:
        r = subprocess.run([JAVA, '-version'], capture_output=True, text=True, timeout=30)
        txt = (r.stderr or '') + (r.stdout or '')
        return '25.' in txt
    except Exception:
        return False

def fetch(url, dest):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=300) as r, open(dest, 'wb') as f:
        got = 0
        while True:
            c = r.read(1024 * 1024)
            if not c:
                break
            f.write(c)
            got += len(c)
    return got

def extract(tarfile):
    subprocess.run('rm -rf /opt/java25 && mkdir -p /opt/java25', shell=True)
    subprocess.run('tar xzf ' + tarfile + ' -C /opt/java25 --strip-components=1', shell=True, check=True)
    return java_ok()

if java_ok():
    print('✅ Java 25 pehle se ready hai')
else:
    done = False

    # 1 + 2) Adoptium (official Temurin) — JRE phir JDK
    for name, url, fname in [
        ('Adoptium JRE', 'https://api.adoptium.net/v3/binary/latest/25/ga/linux/x64/jre/hotspot/normal/eclipse', 'jre25.tar.gz'),
        ('Adoptium JDK', 'https://api.adoptium.net/v3/binary/latest/25/ga/linux/x64/jdk/hotspot/normal/eclipse', 'jdk25.tar.gz'),
    ]:
        if done:
            break
        try:
            print('⬇️ Java 25 (' + name + ') download...')
            mb = fetch(url, fname) / 1024 / 1024
            print('   mila: %.0f MB — extract...' % mb)
            if extract(fname):
                print('✅ Java 25 ready (' + name + ')')
                done = True
        except Exception as e:
            print('⚠️ ' + name + ' fail:', str(e)[:120])

    # 3) Drive cache
    if not done:
        try:
            if DRIVE_BASE:
                cached = os.path.join(DRIVE_BASE, 'jre25.tar.gz')
                if os.path.exists(cached):
                    print('⬇️ Java Drive cache se aa raha hai...')
                    shutil.copy(cached, 'jre25.tar.gz')
                    if extract('jre25.tar.gz'):
                        print('✅ Java 25 ready (Drive cache)')
                        done = True
        except Exception as e:
            print('⚠️ Drive cache fail:', str(e)[:120])

    # 4) Azul Zulu
    if not done:
        try:
            print('⬇️ Azul (last fallback) try...')
            html = urllib.request.urlopen(urllib.request.Request('https://cdn.azul.com/zulu/bin/', headers=UA), timeout=120).read().decode('utf-8', 'ignore')
            cands = sorted(set(re.findall(r'zulu25\.[0-9.]+-ca-(?:jre|jdk)25\.[0-9.]+-linux_x64\.tar\.gz', html)))
            if cands:
                print('   mila:', cands[-1])
                fetch('https://cdn.azul.com/zulu/bin/' + cands[-1], 'zulu25.tar.gz')
                if extract('zulu25.tar.gz'):
                    print('✅ Java 25 ready (Azul)')
                    done = True
        except Exception as e:
            print('⚠️ Azul fail:', str(e)[:120])

    # Success hua to Drive me cache save karo (agle session ke liye)
    if done:
        try:
            if DRIVE_BASE:
                cached = os.path.join(DRIVE_BASE, 'jre25.tar.gz')
                if not os.path.exists(cached):
                    for f in ('jre25.tar.gz', 'zulu25.tar.gz'):
                        if os.path.exists(f):
                            shutil.copy(f, cached)
                            print('💾 Java cache Drive me save')
                            break
        except Exception as e:
            print('ℹ️ cache save skip:', str(e)[:80])
    else:
        print('❌ Java install fail — is cell ko dobara Run karo (internet issue)')

# java command ko v25 banao + verify
try:
    if java_ok():
        if os.path.lexists('/usr/local/bin/java'):
            os.remove('/usr/local/bin/java')
        os.symlink(JAVA, '/usr/local/bin/java')
        r = subprocess.run([JAVA, '-version'], capture_output=True, text=True)
        ver = ((r.stderr or r.stdout) or '').strip().splitlines()[0] if (r.stderr or r.stdout) else ''
        print('Java version:', ver)
        print('✅ Java 25 READY!')
except Exception as e:
    print('⚠️ symlink issue (chalega):', e)

In [ ]:
# 📦 PAPER server.jar — 2-step fallback (API -> Drive cache)
import urllib.request, json, os, shutil

UA = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

def jar_size():
    try:
        return os.path.getsize('server.jar')
    except Exception:
        return 0

# 1) Latest build API se
try:
    req = urllib.request.Request('https://fill.papermc.io/v3/projects/paper/versions/26.3/builds', headers=UA)
    latest = json.loads(urllib.request.urlopen(req, timeout=30).read())[0]
    print('📦 Paper 26.3 latest build: #' + str(latest['id']))
    dl = latest['downloads']['server:default']['url']
    req2 = urllib.request.Request(dl, headers=UA)
    with urllib.request.urlopen(req2, timeout=180) as r, open('server.jar', 'wb') as f:
        got = 0
        while True:
            c = r.read(1024 * 1024)
            if not c:
                break
            f.write(c)
            got += len(c)
    print('   download: %.0f MB' % (got / 1024 / 1024))
except Exception as e:
    print('⚠️ API download fail:', str(e)[:140])

# 2) Fail/chhoti file ho to Drive cache se
if jar_size() < 40 * 1024 * 1024:
    got_cache = False
    try:
        if DRIVE_BASE:
            cached = os.path.join(DRIVE_BASE, 'server.jar')
            if os.path.exists(cached) and os.path.getsize(cached) > 40 * 1024 * 1024:
                shutil.copy(cached, 'server.jar')
                got_cache = True
                print('✅ server.jar Drive cache se aa gaya')
    except Exception as e:
        print('⚠️ Drive cache fail:', str(e)[:120])
    if not got_cache:
        try:
            if os.path.exists('server.jar') and os.path.getsize('server.jar') < 40 * 1024 * 1024:
                os.remove('server.jar')
        except Exception:
            pass

# 3) Fresh aaya ho to Drive me cache update
if jar_size() > 40 * 1024 * 1024:
    try:
        if DRIVE_BASE:
            cached = os.path.join(DRIVE_BASE, 'server.jar')
            if (not os.path.exists(cached)) or os.path.getsize(cached) != os.path.getsize('server.jar'):
                shutil.copy('server.jar', cached)
                print('💾 server.jar Drive cache updated')
    except Exception:
        pass
    print('✅ server.jar ready: %.1f MB' % (jar_size() / 1024 / 1024))
else:
    print('❌ server.jar nahi mila — is cell ko dobara Run karo (internet issue)')

# Config files (hamesha likho)
open('eula.txt', 'w').write('eula=true\n')
open('server.properties', 'w').write(
    'motd=Colab SMP - Dosto ka Adda\n'
    'max-players=10\n'
    'difficulty=normal\n'
    'gamemode=survival\n'
    'view-distance=8\n'
    'simulation-distance=6\n'
    'spawn-protection=0\n')
print('✅ eula.txt + server.properties ready')

In [ ]:
# 🚀 ngrok v3 (PERMANENT panel link ke liye)
!wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz -O ngrok.tgz && tar xzf ngrok.tgz
!./ngrok --version
print('✅ Ngrok v3 ready!')

In [ ]:
# 🚀 ngrok config — authtoken Colab 🔑 secret se aata hai
import os

if not globals().get('NGROK_TOKEN'):
    print('⚠️ NGROK_TOKEN nahi mila — ngrok skip (sirf backup link chalega)')
    print('   Fix: sidebar 🔑 me NGROK_TOKEN daalo, phir Run all dobara')
else:
    open('ngrok.yml', 'w').write(
        'version: 3\n'
        'agent:\n'
        '  authtoken: ' + NGROK_TOKEN + '\n'
        'tunnels:\n'
        '  panel:\n'
        '    proto: http\n'
        '    addr: 8081\n')
    print('✅ ngrok.yml ready (permanent link config)')

In [ ]:
# 🛟 cloudflared (BACKUP panel link — purane session se ZERO takrav)
import os, subprocess

if not os.path.exists('cloudflared') or os.path.getsize('cloudflared') < 5 * 1024 * 1024:
    print('⬇️ cloudflared download (backup tunnel, ~40 MB)...')
    subprocess.run("curl -sL --retry 3 --retry-delay 2 -f https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o cloudflared", shell=True)
    subprocess.run('chmod +x cloudflared', shell=True)

if os.path.exists('cloudflared') and os.path.getsize('cloudflared') > 5 * 1024 * 1024:
    print('✅ cloudflared ready (backup link)')
else:
    print('⚠️ cloudflared download fail — backup link nahi milega, PERMANENT (ngrok) chalega')

In [ ]:
# 🤖 SELF-HEALING AGENT — server + panel + ngrok + cloudflared + playit sab auto-restart
import subprocess, time, threading, os, re, json, urllib.request
from datetime import datetime

JAVA = '/opt/java25/bin/java'   # Paper 26.3 = Java 25 mandatory!
PLAYIT_SECRET = globals().get('PLAYIT_SECRET') or ''   # Colab 🔑 se aata hai
DRIVE_BASE = globals().get('DRIVE_BASE')
lock = threading.Lock()

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

def jar_ok():
    try:
        return os.path.getsize('server.jar') > 40 * 1024 * 1024
    except Exception:
        return False

def is_running():
    if os.path.exists('STOP_FLAG'):
        return True   # Panel se STOP kiya gaya hai — watchdog restart na kare
    r = subprocess.run(['screen', '-ls'], capture_output=True, text=True)
    return 'mcserver' in r.stdout

def service_alive(pattern, exact=False):
    try:
        cmd = ['pgrep', '-x', pattern] if exact else ['pgrep', '-f', pattern]
        return subprocess.run(cmd, capture_output=True).returncode == 0
    except Exception:
        return False

def start_server():
    with lock:
        if is_running():
            return
        if not jar_ok():
            log('❌ server.jar nahi hai — server.jar wala cell dobara Run karo!')
            return
        if not os.path.exists(JAVA):
            log('❌ Java 25 nahi hai — Java wala cell dobara Run karo!')
            return
        log('🚀 Minecraft server start ho raha hai (Java 25)...')
        subprocess.run(['screen', '-S', 'mcserver', '-X', 'quit'], capture_output=True)
        subprocess.run(['screen', '-wipe'], capture_output=True)
        subprocess.Popen(['screen', '-dmS', 'mcserver', 'bash', '-c',
                          JAVA + ' -Xmx2G -Xms2G -jar server.jar nogui 2>&1 | tee server_console.log'])
        time.sleep(15)
        if is_running():
            log('✅ Server chal gaya! (world load hone me 1-2 min lag sakte hain)')
        else:
            log('❌ Server FAIL hua! Asli error:')
            print('=' * 45)
            try:
                print(open('server_console.log').read()[-1300:])
            except Exception as e:
                print('console log nahi mila:', e)
            print('=' * 45)

def ensure_services():
    if os.path.exists('panel.py') and not service_alive('panel.py'):
        log('🔄 Panel band tha — restart')
        subprocess.Popen(['python3', 'panel.py'],
                         stdout=open('panel.log', 'w'), stderr=subprocess.STDOUT)
    if os.path.exists('ngrok') and os.path.exists('ngrok.yml') and not service_alive('ngrok', exact=True):
        log('🔄 ngrok restart (permanent link retry)')
        subprocess.Popen(['./ngrok', 'start', '--all', '--config', 'ngrok.yml', '--log=ngrok.log'],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if os.path.exists('cloudflared') and not service_alive('cloudflared'):
        log('🔄 cloudflared restart (backup link)')
        subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8081', '--no-autoupdate', '--protocol', 'http2'],
                         stdout=open('cloudflared.log', 'w'), stderr=subprocess.STDOUT)
    if PLAYIT_SECRET and os.path.exists('playit') and not service_alive('playit', exact=True):
        log('🔄 playit restart')
        subprocess.Popen(['./playit', '--secret', PLAYIT_SECRET, '--log-path', 'playit.log'],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

_url_state = {'ngrok': None, 'cf': None}

def url_monitor():
    u = None
    try:
        d = json.loads(urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=4).read())
        urls = [t['public_url'] for t in d.get('tunnels', []) if t.get('public_url')]
        u = urls[0] if urls else None
    except Exception:
        u = None
    if u:
        try:
            open('NGROK_URL', 'w').write(u)
        except Exception:
            pass
        if _url_state['ngrok'] != u:
            _url_state['ngrok'] = u
            log('🎉 PERMANENT panel link LIVE: ' + u)
    else:
        if _url_state['ngrok']:
            _url_state['ngrok'] = ''
            log('⚠️ Permanent link filhaal nahi mila (purana session?) — backup link chal raha hai, permanent KHUD aa jayega')
        try:
            if os.path.exists('NGROK_URL'):
                os.remove('NGROK_URL')
        except Exception:
            pass
    c = None
    try:
        if os.path.exists('cloudflared.log'):
            txt = open('cloudflared.log', errors='ignore').read()[-6000:]
            m = [x for x in re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', txt) if not x.startswith('https://api.')]
            c = m[-1] if m else None
    except Exception:
        c = None
    if c:
        try:
            open('CF_URL', 'w').write(c)
        except Exception:
            pass
        if _url_state['cf'] != c:
            _url_state['cf'] = c
            log('🛟 Backup panel link: ' + c)
    else:
        if _url_state['cf']:
            _url_state['cf'] = ''
            log('⚠️ Backup link gaya — cloudflared restart hoga (naya link ban jayega)')

def backup_loop():
    while True:
        time.sleep(300)
        try:
            if DRIVE_BASE and os.path.isdir(DRIVE_BASE):
                n = 0
                for d in ('world', 'world_nether', 'world_the_end'):
                    src = '/content/' + d
                    if os.path.isdir(src):
                        subprocess.run(['rsync', '-a', '--delete', src + '/', os.path.join(DRIVE_BASE, d) + '/'],
                                       capture_output=True, timeout=180)
                        n += 1
                if n:
                    open('LAST_BACKUP', 'w').write(datetime.now().strftime('%H:%M:%S'))
        except Exception:
            pass

_playit_warn = {'t': 0}

def playit_conflict_check():
    try:
        if not os.path.exists('playit.log'):
            return
        txt = open('playit.log', errors='ignore').read()[-8000:]
        if txt.count('session expired') >= 2:
            now = time.time()
            if now - _playit_warn['t'] > 7200:
                _playit_warn['t'] = now
                log('ℹ️ Lagta hai koi PURANA session bhi playit pe zinda hai — jab tak wo rahega, friends kabhi purane server pe ja sakte hain. Wo khud band hote hi sab theek.')
    except Exception:
        pass

def watchdog():
    time.sleep(45)
    while True:
        try:
            if not is_running():
                log('⚠️ Server down! Restart...')
                start_server()
            ensure_services()
            url_monitor()
            playit_conflict_check()
        except Exception as e:
            log('watchdog err: ' + str(e)[:120])
        time.sleep(30)

# Java confirm
r = subprocess.run([JAVA, '-version'], capture_output=True, text=True)
print('Java:', ((r.stderr or r.stdout).strip().splitlines()[0]) if (r.stderr or r.stdout) else '❌ NAHI MILA (Java wala cell dobara Run karo)')

if '_watchdog_started' not in globals():
    globals()['_watchdog_started'] = True
    threading.Thread(target=watchdog, daemon=True).start()
    threading.Thread(target=backup_loop, daemon=True).start()
    log('🤖 Self-healing agent ON (server + panel + ngrok + cloudflared + playit)')

start_server()

In [ ]:
%%writefile panel.py
"""Minecraft Colab Panel v3.0 — Start/Stop/Restart + Console + Live Log + Services + double panel links (ngrok permanent + cloudflared backup)."""
from flask import Flask, jsonify, request
import subprocess, os, threading, time
from datetime import datetime

app = Flask(__name__)

JAVA = '/opt/java25/bin/java'
FLAG = 'STOP_FLAG'

def is_running():
    r = subprocess.run(['screen', '-ls'], capture_output=True, text=True)
    return 'mcserver' in r.stdout

def do_start():
    if is_running():
        if os.path.exists(FLAG):
            os.remove(FLAG)
        return
    subprocess.run(['screen', '-S', 'mcserver', '-X', 'quit'], capture_output=True)
    subprocess.run(['screen', '-wipe'], capture_output=True)
    subprocess.Popen(['screen', '-dmS', 'mcserver', 'bash', '-c',
                      JAVA + ' -Xmx2G -Xms2G -jar server.jar nogui 2>&1 | tee server_console.log'])
    if os.path.exists(FLAG):
        os.remove(FLAG)

def do_stop():
    open(FLAG, 'w').write('1')
    subprocess.run(['screen', '-S', 'mcserver', '-X', 'stuff', 'stop\n'])

def do_restart():
    open(FLAG, 'w').write('1')
    subprocess.run(['screen', '-S', 'mcserver', '-X', 'stuff', 'stop\n'])
    for _ in range(20):
        time.sleep(3)
        if not is_running():
            break
    subprocess.run(['screen', '-S', 'mcserver', '-X', 'quit'], capture_output=True)
    subprocess.run(['screen', '-wipe'], capture_output=True)
    subprocess.Popen(['screen', '-dmS', 'mcserver', 'bash', '-c',
                      JAVA + ' -Xmx2G -Xms2G -jar server.jar nogui 2>&1 | tee server_console.log'])
    if os.path.exists(FLAG):
        os.remove(FLAG)

PAGE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>MC Panel</title>
<style>
body{font-family:system-ui,sans-serif;background:#0f172a;color:#e2e8f0;margin:0;padding:16px}
.wrap{max-width:520px;margin:0 auto}
h1{font-size:20px;text-align:center;margin:8px 0 16px}
.card{background:#1e293b;border-radius:14px;padding:16px;margin:12px 0}
.dot{display:inline-block;width:12px;height:12px;border-radius:50%;background:#ef4444;margin-right:8px}
.dot.on{background:#22c55e}
button{border:0;border-radius:10px;padding:12px 14px;font-size:15px;font-weight:600;cursor:pointer;margin:4px 2px}
.b-start{background:#22c55e;color:#052e16}
.b-stop{background:#ef4444;color:#fff}
.b-restart{background:#f59e0b;color:#451a03}
.b-send{background:#3b82f6;color:#fff;width:100%}
input{width:100%;box-sizing:border-box;padding:12px;border-radius:10px;border:1px solid #334155;background:#0f172a;color:#fff;font-size:15px}
pre{background:#020617;border-radius:10px;padding:12px;font-size:12px;overflow:auto;max-height:260px;white-space:pre-wrap;margin:0}
.muted{color:#94a3b8;font-size:12px}
.center{text-align:center}
.lk a{color:#7dd3fc;word-break:break-all;font-size:13px}
</style>
</head>
<body>
<div class="wrap">
<h1>🎮 Minecraft Panel v3</h1>
<div class="card">
  <div style="font-size:18px;font-weight:700"><span class="dot" id="dot"></span><span id="st">Check ho raha...</span></div>
  <div class="muted" id="tm"></div>
  <div class="muted" id="svc"></div>
</div>
<div class="card">
  <div style="font-weight:700;margin-bottom:6px">🌐 Panel Links</div>
  <div class="muted" style="margin-bottom:4px">Permanent (bookmark karo):</div>
  <div id="png" class="lk">check...</div>
  <div class="muted" style="margin:10px 0 4px">Backup (emergency — ye hamesha chalta hai):</div>
  <div id="pcf" class="lk">check...</div>
</div>
<div class="card center">
  <button class="b-start" onclick="ctl('start')">▶️ Start</button>
  <button class="b-stop" onclick="ctl('stop')">⏹️ Stop</button>
  <button class="b-restart" onclick="ctl('restart')">🔄 Restart</button>
</div>
<div class="card">
  <input id="cmd" placeholder="Command: say hello / time set day">
  <div style="margin-top:8px"><button class="b-send" onclick="send()">📨 Send Command</button></div>
</div>
<div class="card">
  <div style="font-weight:700;margin-bottom:8px">📜 Server Log (live)</div>
  <pre id="log">loading...</pre>
</div>
<div class="muted center">Auto-refresh 5 sec • Server chal raha hai to dot GREEN hoga</div>
</div>
<script>
async function refresh(){
 try{
  const s = await (await fetch('/api/status')).json();
  document.getElementById('dot').className = 'dot' + (s.running ? ' on' : '');
  document.getElementById('st').textContent = s.running ? 'Server ONLINE ✅' : 'Server OFFLINE ❌';
  document.getElementById('tm').textContent = 'Last check: ' + s.time;
  document.getElementById('svc').textContent = 'playit: ' + (s.playit ? '✅' : '❌') +
    ' | ngrok: ' + (s.ngrok ? '✅' : '❌') + ' | cloudflared: ' + (s.cloudflared ? '✅' : '❌') +
    ' | world backup: ' + (s.backup || '-');
  document.getElementById('png').innerHTML = s.url_ngrok
    ? ('<a href="' + s.url_ngrok + '">' + s.url_ngrok + '</a> ✅ live')
    : '⏳ abhi busy hai (purana session) — kuch nahi karna, khud LIVE ho jayega';
  document.getElementById('pcf').innerHTML = s.url_cf
    ? ('<a href="' + s.url_cf + '">' + s.url_cf + '</a> ✅ live')
    : '⏳ starting... (5-10 sec baad refresh hoga)';
  const l = await (await fetch('/api/log')).json();
  const el = document.getElementById('log');
  el.textContent = l.log;
  el.scrollTop = el.scrollHeight;
 }catch(e){
  document.getElementById('st').textContent = '⚠️ Panel error';
 }
}
async function ctl(c){
  if(c === 'stop' && !confirm('Server band karna hai?')) return;
  await fetch('/api/control',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({command:c})});
  refresh();
}
async function send(){
  const v = document.getElementById('cmd').value;
  if(!v) return;
  await fetch('/api/command',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({command:v})});
  document.getElementById('cmd').value = '';
  refresh();
}
refresh();
setInterval(refresh, 5000);
</script>
</body>
</html>
"""

@app.route('/')
def index():
    return PAGE

def proc_alive(name):
    try:
        return subprocess.run(['pgrep', '-x', name], capture_output=True).returncode == 0
    except Exception:
        return False

def proc_alive_f(pattern):
    try:
        return subprocess.run(['pgrep', '-f', pattern], capture_output=True).returncode == 0
    except Exception:
        return False

def read_file(name):
    try:
        return open(name, errors='ignore').read().strip()
    except Exception:
        return ''

@app.route('/api/status')
def status():
    backup = read_file('LAST_BACKUP') or '-'
    return jsonify({'running': is_running(),
                    'playit': proc_alive('playit'),
                    'ngrok': proc_alive('ngrok'),
                    'cloudflared': proc_alive_f('cloudflared'),
                    'backup': backup,
                    'url_ngrok': read_file('NGROK_URL'),
                    'url_cf': read_file('CF_URL'),
                    'time': datetime.now().strftime('%H:%M:%S')})

@app.route('/api/log')
def log():
    try:
        txt = open('server_console.log', errors='ignore').read()[-1500:]
    except Exception:
        txt = '(server_console.log abhi nahi hai)'
    return jsonify({'log': txt})

@app.route('/api/control', methods=['POST'])
def control():
    c = (request.json or {}).get('command', '')
    if c == 'start':
        threading.Thread(target=do_start, daemon=True).start()
    elif c == 'stop':
        do_stop()
    elif c == 'restart':
        threading.Thread(target=do_restart, daemon=True).start()
    return jsonify({'ok': True})

@app.route('/api/command', methods=['POST'])
def command():
    c = (request.json or {}).get('command', '')
    if c:
        subprocess.run(['screen', '-S', 'mcserver', '-X', 'stuff', c + '\n'])
    return jsonify({'ok': True})

app.run(host='0.0.0.0', port=8081, threaded=True)

In [ ]:
# 🌐 Panel + DONO tunnels start — ab panel KABHI dead nahi milega!
import subprocess, time, urllib.request, json, os, re

def alive(pattern, exact=False):
    cmd = ['pgrep', '-x', pattern] if exact else ['pgrep', '-f', pattern]
    try:
        return subprocess.run(cmd, capture_output=True).returncode == 0
    except Exception:
        return False

# Panel
if not alive('panel.py'):
    subprocess.Popen(['python3', 'panel.py'], stdout=open('panel.log', 'w'), stderr=subprocess.STDOUT)
    print('🌐 Panel start (port 8081)')
time.sleep(3)
try:
    urllib.request.urlopen('http://127.0.0.1:8081/', timeout=5)
    print('✅ Panel v3 live hai (local)')
except Exception as e:
    print('⚠️ Panel local check:', e)

# Tunnel 1: ngrok — PERMANENT link
if os.path.exists('ngrok.yml') and not alive('ngrok', exact=True):
    subprocess.Popen(['./ngrok', 'start', '--all', '--config', 'ngrok.yml', '--log=ngrok.log'],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('🚀 ngrok start (permanent link)')

# Tunnel 2: cloudflared — BACKUP link (kisi purane session se koi takrav nahi!)
if not alive('cloudflared'):
    subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8081', '--no-autoupdate', '--protocol', 'http2'],
                     stdout=open('cloudflared.log', 'w'), stderr=subprocess.STDOUT)
    print('🛟 cloudflared start (backup link)')

print('⏳ Links ready hone ka wait (~30 sec)...')
NG = ''
CF = ''
for i in range(12):
    time.sleep(3)
    if not NG:
        try:
            d = json.loads(urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=3).read())
            if d.get('tunnels'):
                NG = d['tunnels'][0]['public_url']
        except Exception:
            pass
    if not CF and os.path.exists('cloudflared.log'):
        try:
            txt = open('cloudflared.log', errors='ignore').read()
            m = [x for x in re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', txt) if not x.startswith('https://api.')]
            if m:
                CF = m[-1]
        except Exception:
            pass
    if NG and CF:
        break

if NG:
    open('NGROK_URL', 'w').write(NG)
if CF:
    open('CF_URL', 'w').write(CF)

print()
print('═' * 58)
if NG:
    print('🌐 PERMANENT PANEL URL (bookmark karo — hamesha yahi):')
    print('   ' + NG)
else:
    if os.path.exists('ngrok.yml'):
        print('🌐 PERMANENT PANEL URL (abhi busy):')
        print('   https://tidal-chug-rasping.ngrok-free.dev')
        print('   ⏳ Ye kisi purane session ke paas hai — wo band hote hi')
        print('      KHUD LIVE ho jayega. Tumhe kuch nahi karna!')
    else:
        print('🌐 Permanent link OFF (NGROK_TOKEN 🔑 me daalo) — backup link chal raha hai')
print()
if CF:
    print('🛟 BACKUP PANEL URL (ye hamesha chalta hai):')
    print('   ' + CF)
else:
    print('🛟 Backup link 1 min me ban jayega — panel page pe bhi dikhega')
print('═' * 58)
best = NG or CF
if best:
    print('👉 ABHI YAHI KHOLO: ' + best)
else:
    print('👉 1 min ruko phir panel page kholo — link khud dikh jayega')
print('📱 ngrok pe pehli baar "Visit Site" tap karna; cloudflare pe seedha khulta hai')

In [ ]:
# 🎮 playit.gg (Minecraft — FREE, no card! secret Colab 🔑 se aata hai)
import subprocess, time, os

if not globals().get('PLAYIT_SECRET'):
    print('❌ PLAYIT_SECRET nahi mila — Minecraft tab tak nahi chalega!')
    print('   Fix: sidebar 🔑 → Add new secret → Name: PLAYIT_SECRET')
    print('   Value chat me hai | "Notebook access" toggle ON!')
    print('   Phir ye cell dobara Run karo. (Panel ka backup link phir bhi chalega)')
else:
    if not os.path.exists('playit') or os.path.getsize('playit') < 3 * 1024 * 1024:
        print('⬇️ playit download ho raha hai (~6 MB)...')
        subprocess.run("curl -sL --retry 3 -f https://github.com/playit-cloud/playit-agent/releases/download/v1.0.10/playit-linux-amd64 -o playit && chmod +x playit",
                       shell=True)
    print('✅ playit binary ready')

    subprocess.run("pkill -x playit || true", shell=True)
    time.sleep(1)
    subprocess.Popen(['./playit', '--secret', PLAYIT_SECRET, '--log-path', 'playit.log'],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('🚀 playit agent start (wait 10 sec)...')
    time.sleep(10)

    if os.path.exists('playit.log'):
        print('--- playit.log ---')
        print(open('playit.log').read()[-600:])

    print()
    print('ℹ️ Log me WARN / SessionNotSetup aaye to IGNORE karo — normal hai.')
    print('   Bas "playit connected ... verified" dikhe to agent OK!')
    print()
    print('═' * 52)
    print('📱 playit.gg → Tunnels → Add Tunnel (agar abhi tak nahi banaya):')
    print('   → Type: Minecraft (Java)')
    print('   → Local: 127.0.0.1  Port: 25565')
    print('   → Create → jo address mile WO DOSTO KO DO! 🎮')
    print('   (Address SAME rehta hai — ek baar bana lo, hamesha ke liye)')
    print('═' * 52)

In [ ]:
# 🔒 KEEPALIVE — Colab session ko jaagta rakho (fallback: JS click + busy loop)
# ⚠️ YE CELL HAMESHA CHALTA RAHEGA — ye normal hai! (Rokna ho to ■ Stop dabao)
import time, os
from datetime import datetime

try:
    from google.colab import output
    output.eval_js('''
    (function(){
      if (window.__mcKeepAlive) return "already-on";
      window.__mcKeepAlive = setInterval(function(){
        try {
          var btn = document.querySelector("colab-connect-button");
          if (btn) btn.click();
        } catch(e){}
      }, 60000);
      return "on";
    })();
    ''')
    print('🔒 Browser keep-alive ON')
except Exception as e:
    print('ℹ️ JS keepalive skip:', e)

print('🔒 Session keep-alive ON — Colab ko idle hone nahi dega. (Tab khula rakho!)')
n = 0
while True:
    time.sleep(300)
    n += 1
    line = '💚 Alive #' + str(n) + ' — ' + datetime.now().strftime('%H:%M:%S')
    if n % 3 == 0:
        try:
            ng = open('NGROK_URL').read().strip() if os.path.exists('NGROK_URL') else ''
            cf = open('CF_URL').read().strip() if os.path.exists('CF_URL') else ''
            print(line + ' | 🌐 ' + (ng or 'permanent ⏳ (busy — backup chal raha hai)') + ' | 🛟 ' + (cf or '-'))
        except Exception:
            print(line)
    else:
        print(line)

---

## 📝 Notes (v9)

| Cheez | Detail |
|---|---|
| **🌐 Panel (permanent)** | `https://tidal-chug-rasping.ngrok-free.dev` — bookmark kar lo! |
| **🛟 Panel (backup)** | Cloudflare link — agar permanent busy ho to ye hamesha chalega |
| **🎮 Minecraft** | playit.gg address (ek baar tunnel banao — address hamesha same) |
| **🎨 Panel features** | Start/Stop/Restart + commands + LIVE log + services + dono links |
| **🤖 Self-healing** | server/panel/ngrok/cloudflared/playit — har 30 sec check, mare to khud uthao |
| **☕ Java** | 4 fallbacks: Adoptium JRE → JDK → Drive cache → Azul |
| **📦 server.jar** | 2 fallbacks: Paper API → Drive cache |
| **💾 World backup** | Har 5 min Google Drive (`MyDrive/MC_Colab`) — session reset pe bhi safe |
| **🔒 Keepalive** | Last cell hamesha chalega (JS click + busy loop) — tab khula rakho |
| **⏰ Colab limit** | Free VM ~12 ghante — wapas aake **Run all** (world Drive se wapas aayega) |
| **⛔ Server band** | Panel me **Stop** button (watchdog restart nahi karega) |
| **🧯 Purana session** | Zinda ho to bhi koi dikkat nahi — backup link chalega, permanent khud kabza karega |
| **🔑 Secrets** | Colab ke 🔑 sidebar me (1 baar daalo, hamesha yaad) — ye file PUBLIC hai, phir bhi safe ✅ |